# OpenSearch — service test

Tests `OpenSearchClient`: health, index stats, BM25 search, hybrid search.

**Prereqs:** `docker compose up -d opensearch` + index populated (run the DAG or `06_indexing.ipynb`). 
**Note:** plain HTTP on localhost:9200 — security plugin is disabled.

In [ ]:
from dotenv import load_dotenv
load_dotenv()

from src.services.opensearch.factory import make_opensearch_client

os_client = make_opensearch_client()
print('healthy:', os_client.health_check())

In [ ]:
# How many chunks / unique papers are indexed?
count = os_client.client.count(index=os_client.index_name)['count']
aggs = os_client.client.search(
    index=os_client.index_name,
    body={'size': 0, 'aggs': {'papers': {'cardinality': {'field': 'arxiv_id'}}}},
)
print('total chunks :', count)
print('unique papers:', aggs['aggregations']['papers']['value'])

In [ ]:
# BM25-only search (keyword matching via inverted index)
results = os_client.search_unified(query='attention mechanism', size=3, from_=0, use_hybrid=False, min_score=0.0)
for h in results.get('hits', []):
    print(f"{h.get('arxiv_id'):14} | {h.get('chunk_text', '')[:80]}")

In [ ]:
# Hybrid search (BM25 + k-NN + RRF fusion) — needs a query embedding
from src.services.embeddings.factory import make_openai_embeddings_client

embedder = make_openai_embeddings_client()
qvec = await embedder.embed_text('attention mechanism')

results = os_client.search_unified(
    query='attention mechanism', query_embedding=qvec,
    size=3, from_=0, use_hybrid=True, min_score=0.0,
)
for h in results.get('hits', []):
    print(f"{h.get('arxiv_id'):14} | {h.get('chunk_text', '')[:80]}")